In [1]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

image = load_image("../20231077135.jpg")

/home/patil139/scratch/adversarial-simulation/.venv/lib/python3.12/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (134640000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


In [2]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vit7b16-pretrain-lvd1689m", image_size=512, patch_size=16)
model = AutoModel.from_pretrained(
    "facebook/dinov3-vit7b16-pretrain-lvd1689m",
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [3]:
patch_size = model.config.patch_size
img_size = 1024

inputs = processor(images=image, size={"height": img_size, "width": img_size}, return_tensors="pt").to(model.device)

batch_size, _, img_height, img_width = inputs.pixel_values.shape
num_patches_height, num_patches_width = img_height // patch_size, img_width // patch_size
num_patches_flat = num_patches_height * num_patches_width

print(inputs.pixel_values.shape)
assert inputs.pixel_values.shape == (batch_size, 3, num_patches_height * patch_size, num_patches_width * patch_size)

torch.Size([1, 3, 1024, 1024])


In [4]:
with torch.inference_mode():
  outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print(last_hidden_states.shape)
assert last_hidden_states.shape == (batch_size, 1 + model.config.num_register_tokens + num_patches_flat, model.config.hidden_size)

cls_token = last_hidden_states[:, 0, :]
patch_features_flat = last_hidden_states[:, 1 + model.config.num_register_tokens:, :]
patch_features = patch_features_flat.unflatten(1, (num_patches_height, num_patches_width))
print(cls_token.shape)
print(patch_features.shape)
assert cls_token.shape == (batch_size, model.config.hidden_size)
assert patch_features.shape == (batch_size, num_patches_height, num_patches_width, model.config.hidden_size)

torch.Size([1, 4101, 4096])
torch.Size([1, 4096])
torch.Size([1, 64, 64, 4096])
